# FoundML3 — Week X Coding Assignment
## Model Interpretation & Explanatory AI (XGBoost + Kaggle Cardiovascular)

In this notebook you will:
- Train an **XGBoost** model on the **raw Kaggle cardiovascular dataset**.
- Compare **Permutation Feature Importance (PFI)** and **SHAP** global importance.
- Quantify **stability** of explanations across random seeds.
- Compute a **counterfactual** for a rejected/high-risk case under **immutable feature constraints**.

**Deliverable:** numerical answers to the *Challenge Questions* embedded throughout. Enter them into Moodle.

### Dataset
Download the Kaggle dataset (file typically named `cardio_train.csv`) and place it next to this notebook, or update the path in the next cell.

⚠️ Keep everything else (seeds, parameters) unchanged to match Moodle answers.


In [ ]:
# --- Setup (DO NOT MODIFY) ---
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import shap
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED_SPLIT = 123
SEED_BASE = 0
SEEDS_STABILITY = [0, 1, 2, 3, 4]

CSV_PATH = 'cardio_train.csv'  # update only if needed
TARGET_COL = 'cardio'
DROP_COLS = ['id']  # raw Kaggle includes 'id'

RND = np.random.default_rng(0)


## Part 0 — Load data (raw Kaggle)
We keep the raw columns as in the Kaggle file. Minimal cleaning:
- drop `id`
- ensure all columns are numeric


In [ ]:
# Load CSV
df = pd.read_csv(CSV_PATH, sep=';')
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
assert TARGET_COL in df.columns, f"Expected target column {TARGET_COL}"

X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED_SPLIT, stratify=y
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

## Part 1 — Train baseline XGBoost
We use fixed hyperparameters for reproducibility.


In [ ]:
# --- Baseline model (DO NOT MODIFY hyperparameters) ---
def make_xgb(seed: int) -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.6,
        colsample_bytree=0.6,
        reg_lambda=0.1,
        random_state=seed,
        n_jobs=1,
        tree_method='hist',
        eval_metric='logloss'
    )


# Your code here
# - Train the baseline model with seed SEED_BASE
# - Compute:
#   auc_test  (float): ROC-AUC on the test set
#   mean_proba (float): mean predicted probability on the test set
# - Keep the trained model in variable `model`


### ✅ Challenge Questions (Part 1)
**Q1.** Test ROC-AUC of the baseline model (round to 4 decimals): ____  
**Q2.** Mean predicted probability on the test set (round to 4 decimals): ____


## Part 2 — Global explanations: PFI vs SHAP
We compute:
- **Permutation Feature Importance** on the test set (scoring = ROC-AUC).
- **SHAP global importance** as mean absolute SHAP value on the test set.

Then we compare rankings.


In [ ]:
# --- PFI (Permutation Feature Importance) ---
# Use n_repeats=10, random_state=777, scoring='roc_auc'

# Your code here


In [ ]:
# --- SHAP global importance (mean absolute SHAP) ---
# Use the TreeExplainer function
# Your code here


In [ ]:
# Compare rankings via Spearman correlation
# Your code here


### ✅ Challenge Questions (Part 2)
**Q3.** PFI value for `ap_hi` (AUC decrease; round to 4 decimals): ____  
**Q4.** PFI value for `weight` (round to 4 decimals): ____  
**Q5.** mean(|SHAP|) for `ap_hi` (round to 4 decimals): ____  
**Q6.** Spearman correlation between PFI and SHAP rankings (round to 4 decimals): ____


## Part 3 — Stability of explanations across seeds
We retrain the same model with different seeds and quantify variability of:
- test AUC
- SHAP global importance vectors

We will compute:
- AUC standard deviation
- SHAP standard deviation for a selected feature
- cosine similarity between global SHAP vectors for two seeds
- feature with the largest coefficient of variation (std/mean)


In [ ]:
# Train models across seeds and collect metrics
# Your code here


In [ ]:
# SHAP variability summaries
# Your code here


In [ ]:
# Cosine similarity between seed 0 and seed 1 global SHAP vectors
# Your code here


### ✅ Challenge Questions (Part 3)
**Q7.** Standard deviation of test AUC across the 5 seeds (round to 4 decimals): ____  
**Q8.** Standard deviation across seeds of SHAP global importance for `gluc` (round to 4 decimals): ____  
**Q9.** Cosine similarity between SHAP global vectors for seed 0 and seed 1 (round to 4 decimals): ____  
**Q10.** Which feature has the largest coefficient of variation (std/mean) of SHAP global importance?
Choose one option in Moodle: 
- A. ap_hi  
- B. ap_lo  
- C. cholesterol  
- D. smoke  
- E. active


## Part 4 — Tutorial on counterfactual explanation with immutable features
We construct a counterfactual \(x'\) for a **high-risk** test instance.

Immutable features (do not change):
- `age`, `gender`, `height`

Mutable features (allowed to change):
- `weight`, `ap_hi`, `ap_lo`, `cholesterol`, `gluc`, `smoke`, `alco`, `active`

We use a simple, deterministic *constrained search* that proposes small edits and keeps the best improvement per unit cost.


In [ ]:
# Choose a fixed test instance index (DO NOT MODIFY)
TEST_INDEX = 0  # index within X_test after train_test_split
x0 = X_test.iloc[[TEST_INDEX]].copy()
p0 = model.predict_proba(x0)[0, 1]
p0

In [ ]:
# Define immutable/mutable feature sets
IMMUTABLE = ['age', 'gender', 'height']
MUTABLE = [c for c in X_train.columns if c not in IMMUTABLE]
MUTABLE

In [ ]:
# Fit a scaler on training data for distance calculations
scaler = StandardScaler()
scaler.fit(X_train)

def scaled_l2(a: pd.DataFrame, b: pd.DataFrame) -> float:
    za = scaler.transform(a)
    zb = scaler.transform(b)
    return float(np.linalg.norm(za - zb))

train_mins = X_train.min()
train_maxs = X_train.max()

In [ ]:
# Deterministic proposal generator for one-step edits
def propose_edits(x: pd.DataFrame) -> list[pd.DataFrame]:
    proposals = []
    row = x.iloc[0]
    # Continuous-ish features: propose +/- small steps (clipped)
    step_map = {
        'weight': 2.0,
        'ap_hi': 5.0,
        'ap_lo': 5.0,
    }
    for f, step in step_map.items():
        if f in x.columns and f in MUTABLE:
            for direction in [-1, 1]:
                x_new = x.copy()
                x_new[f] = np.clip(row[f] + direction * step, train_mins[f], train_maxs[f])
                proposals.append(x_new)
    # Ordinal categorical: cholesterol, gluc in {1,2,3}
    for f in ['cholesterol', 'gluc']:
        if f in x.columns and f in MUTABLE:
            for val in [1, 2, 3]:
                if val != int(row[f]):
                    x_new = x.copy()
                    x_new[f] = val
                    proposals.append(x_new)
    # Binary lifestyle: smoke, alco, active
    for f in ['smoke', 'alco', 'active']:
        if f in x.columns and f in MUTABLE:
            x_new = x.copy()
            x_new[f] = 1 - int(row[f])
            proposals.append(x_new)
    # Enforce immutables unchanged
    for p in proposals:
        for f in IMMUTABLE:
            if f in p.columns:
                p[f] = row[f]
    return proposals

len(propose_edits(x0))

In [ ]:
# Greedy counterfactual search (deterministic)
def find_counterfactual(x_start: pd.DataFrame, target_p: float = 0.5, max_steps: int = 25):
    x_curr = x_start.copy()
    p_curr = model.predict_proba(x_curr)[0, 1]
    for _ in range(max_steps):
        if p_curr < target_p:
            break
        cands = propose_edits(x_curr)
        best = None
        best_score = None
        for x_c in cands:
            p_c = model.predict_proba(x_c)[0, 1]
            # cost = scaled distance from start (encourage small moves)
            cost = scaled_l2(x_start, x_c)
            # objective: reduce probability with small cost
            score = (p_curr - p_c) / (cost + 1e-9)
            if best_score is None or score > best_score:
                best_score = score
                best = (x_c, p_c, cost)
        # if no improvement, stop
        if best is None or best[1] >= p_curr:
            break
        x_curr, p_curr, _ = best
    return x_curr, p_curr

x_cf, p_cf = find_counterfactual(x0, target_p=0.5, max_steps=25)
dist_cf = scaled_l2(x0, x_cf)
p0, p_cf, dist_cf

In [ ]:
# Count changed mutable features (simple discrete check)
diff = (x_cf.iloc[0] != x0.iloc[0])
changed_mutable = [c for c in MUTABLE if diff.get(c, False)]
len(changed_mutable), changed_mutable

## Submission
Submit answers for Q1–Q10 in Moodle.

Tip: Keep your outputs with **4-decimal rounding** to match grading.
